# Notebook 0 — EuroSAT Embeddings + t-SNE

### *Hands-On Session: Earth Embeddings for EO — Retrieval, Discovery, and Change-Oriented Search*

**Instructors:** Fuxun Yu, Rishi Madhok (TerraByte AI)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iceysteel/ESA-NASA-Workshop-2026/blob/main/Day%202/Track%202/Earth-Embeddings-EO/notebook_0_eurosat_embeddings_tsne.ipynb)

---

## Why this notebook?

Before we touch heavyweight Earth-Observation foundation models (AlphaEarth, Prithvi, TerraMind, RS-CLIP), let's anchor the core idea the whole session rests on:

> **The penultimate layer of *any* trained vision model is a usable embedding space — points close together in that space tend to look semantically similar.**

We'll demonstrate this with the smallest possible setup: a ResNet-50 already fine-tuned on EuroSAT (10-class land-cover classification on Sentinel-2 RGB chips). We will:

1. Load the [`cm93/resnet50-eurosat`](https://huggingface.co/cm93/resnet50-eurosat) checkpoint.
2. Run a subset of the EuroSAT test set through it and grab the **2048-dim pre-classifier features**.
3. Sanity-check with a quick **nearest-neighbour retrieval** in embedding space.
4. Project to 2D with **t-SNE** and visualise the cluster structure.

In the following notebooks we'll swap this classifier for proper EO foundation models, build ANN indexes, and run text-to-EO and change-oriented queries on top of the same retrieval pattern you'll see working here.


## Runtime

**Recommended:** `Runtime → Change runtime type → T4 GPU` (free) — full notebook runs in ~1–2 minutes. 
**Fallback:** CPU — ~5–7 minutes on the 2,000-image subset we use below. Still inside the time budget.


In [ ]:
!pip install -q timm datasets


In [ ]:
import time, random
from collections import Counter

import numpy as np
import torch
import timm
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


## 1. Load the EuroSAT test set

EuroSAT is a 10-class land-cover benchmark over Sentinel-2 RGB chips (64×64 px). We pull the **`test` split** of [`timm/eurosat-rgb`](https://huggingface.co/datasets/timm/eurosat-rgb) — a small, parquet-backed mirror that downloads in seconds — and subsample 2,000 images for a predictable runtime.


In [ ]:
ds_full = load_dataset('timm/eurosat-rgb', split='test')
class_names = ds_full.features['label'].names
print('Test split size :', len(ds_full))
print('Classes         :', class_names)

N_SAMPLES = 1500
ds = ds_full.shuffle(seed=SEED).select(range(N_SAMPLES))
print(f'Using {len(ds)} images for this demo.')
print('Per-class counts:', dict(Counter(ds['label'])))


### Peek at the data — one example per class

In [ ]:
examples_by_class = {}
for ex in ds_full:
    if ex['label'] not in examples_by_class:
        examples_by_class[ex['label']] = ex['image']
    if len(examples_by_class) == len(class_names):
        break

fig, axes = plt.subplots(2, 5, figsize=(12, 5.5))
for ax, (lbl, img) in zip(axes.flat, sorted(examples_by_class.items())):
    ax.imshow(img); ax.set_title(class_names[lbl], fontsize=10); ax.axis('off')
plt.suptitle('EuroSAT — one sample per class', y=1.02)
plt.tight_layout(); plt.show()


## 2. Load the pre-trained classifier and strip its head

`timm.create_model(..., num_classes=0)` returns the backbone with the classifier head removed — `model(x)` will now output the **pooled 2048-d feature vector** directly. No forward hooks needed.

We resolve the model's *own* preprocessing config (input size, normalization) via `timm.data.resolve_model_data_config` rather than hardcoding ImageNet stats — the EuroSAT-finetuned checkpoint uses dataset-specific mean/std.


In [ ]:
model = timm.create_model('hf_hub:cm93/resnet50-eurosat', pretrained=True, num_classes=0)
model = model.eval().to(device)

data_cfg = timm.data.resolve_model_data_config(model)
print('Input size :', data_cfg['input_size'])
print('Mean / Std :', data_cfg['mean'], '/', data_cfg['std'])

transform = timm.data.create_transform(**data_cfg, is_training=False)

# Sanity-check: a single forward pass should give us a (1, 2048) vector
dummy = transform(examples_by_class[0].convert('RGB')).unsqueeze(0).to(device)
with torch.inference_mode():
    feat = model(dummy)
print('Embedding shape from one image:', tuple(feat.shape))


## 3. Extract embeddings for the test subset

Simple batched loop in `torch.inference_mode()`. On a T4 this is ~20s; on CPU ~3–5 min.


In [ ]:
BATCH_SIZE = 64

def batched(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i+n]

labels = np.array(ds['label'])
indices = list(range(len(ds)))

embeddings = []
t0 = time.time()
with torch.inference_mode():
    for batch_idx in tqdm(list(batched(indices, BATCH_SIZE)), desc='Embedding'):
        imgs = [transform(ds[i]['image'].convert('RGB')) for i in batch_idx]
        x = torch.stack(imgs).to(device)
        feats = model(x).cpu().numpy()
        embeddings.append(feats)

embeddings = np.concatenate(embeddings, axis=0)
print(f'\nEmbeddings shape: {embeddings.shape}  (wall-clock {time.time()-t0:.1f}s)')


## 4. Sanity check — nearest neighbours in embedding space

If the embedding is meaningful, querying with one image should return images that **look like it** (and usually share its class). This is the same retrieval primitive we'll scale up with ANN indexes in later notebooks.


In [ ]:
rng = np.random.default_rng(SEED)
query_idx = rng.choice(len(embeddings), size=3, replace=False)
sims = cosine_similarity(embeddings[query_idx], embeddings)

K = 5
fig, axes = plt.subplots(3, K + 1, figsize=(2.2 * (K + 1), 6.5))
for row, q in enumerate(query_idx):
    order = np.argsort(-sims[row])
    # drop the query itself (similarity == 1.0)
    neighbours = [j for j in order if j != q][:K]

    axes[row, 0].imshow(ds[int(q)]['image']); axes[row, 0].axis('off')
    axes[row, 0].set_title(f'QUERY\n{class_names[labels[q]]}', fontsize=9, color='C3')
    for col, j in enumerate(neighbours, start=1):
        axes[row, col].imshow(ds[int(j)]['image']); axes[row, col].axis('off')
        axes[row, col].set_title(f'#{col}  {class_names[labels[j]]}\ncos={sims[row, j]:.2f}', fontsize=9)

plt.suptitle('Embedding-space nearest neighbours (cosine similarity)', y=1.02)
plt.tight_layout(); plt.show()


## 5. t-SNE projection of the 2048-dim embeddings → 2D

t-SNE preserves *local* neighbourhood structure — if the embedding genuinely separates classes, we should see coherent islands in the scatter plot.

*Note:* t-SNE distances and absolute coordinates are **not meaningful**; only neighbour groupings are. For retrieval we always operate in the full 2048-dim space.


In [ ]:
t0 = time.time()
emb2d = TSNE(
    n_components=2,
    perplexity=30,
    init='pca',
    random_state=SEED,
    max_iter=1000,
).fit_transform(embeddings)
print(f't-SNE done in {time.time()-t0:.1f}s — shape {emb2d.shape}')


In [ ]:
plt.figure(figsize=(10, 8))
cmap = plt.get_cmap('tab10')
for k, name in enumerate(class_names):
    m = labels == k
    plt.scatter(emb2d[m, 0], emb2d[m, 1], s=10, alpha=0.7, color=cmap(k), label=name)
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=10)
plt.title('EuroSAT test embeddings (ResNet-50 penultimate layer) — t-SNE 2D')
plt.xticks([]); plt.yticks([])
plt.tight_layout(); plt.show()


## 6. What to take away

- **Embeddings ≠ predictions.** We threw away the classifier head and the resulting 2048-dim vectors *still* organise themselves by land-cover semantics. This is the property every retrieval, clustering, and change-detection workflow in this session relies on.
- **Cluster structure tells you about the model, not the truth.** Tight, well-separated clusters (e.g. `SeaLake`, `Forest`, `Residential`) reflect features the network finds easy. Overlapping clusters (e.g. `Pasture` ↔ `HerbaceousVegetation`, `Highway` ↔ `Industrial`) flag **confounders** — semantically distinct classes that share visual texture. Expect the same kinds of confounders in EO foundation-model embeddings, plus extras (cloud cover, season, sensor differences).
- **This model saw labels.** It was *supervised* on EuroSAT, so good clustering on EuroSAT is expected. The next notebooks switch to **self-supervised** and **multimodal-aligned** foundation models that produce useful embedding spaces *without* per-task labels — the actual workhorse pattern for operational EO.
- **Open questions for the rest of the session:**
  - How do these embeddings transfer to a *new* region, sensor, or season? (Domain shift.)
  - How do we scale similarity search from 2,000 vectors to 200M? (ANN indexes — FAISS / ScaNN.)
  - How do we query with **text** instead of an example image? (Multimodal alignment — RS-CLIP.)
  - How do we express *change* — "places that became urban between 2018 and 2022"? (Vector arithmetic on embeddings.)

**On to Notebook 1.**
